# Course 04: Guardrails and Untrusted Content

In this lesson, you will build deterministic guardrails around untrusted content...

In [ ]:
from enum import Enum
from typing import List, Optional, Literal, Set, Type
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from datetime import datetime, timezone, timedelta
import json
import hashlib
import re

## Step 1: Strict Schemas and Policy Constants

In [ ]:
class TrustLevel(Enum):
    TRUSTED = "TRUSTED"
    UNTRUSTED = "UNTRUSTED"
    QUARANTINED = "QUARANTINED"

class Sensitivity(Enum):
    PUBLIC = "PUBLIC"
    INTERNAL = "INTERNAL"
    CONFIDENTIAL = "CONFIDENTIAL"
    RESTRICTED = "RESTRICTED"

class SourceType(Enum):
    USER = "USER"
    RAG_DOCUMENT = "RAG_DOCUMENT"
    WEB = "WEB"
    TOOL_OUTPUT = "TOOL_OUTPUT"

class ContentDisposition(Enum):
    ALLOW_AS_DATA = "ALLOW_AS_DATA"
    QUARANTINE = "QUARANTINE"
    REQUIRE_REVIEW = "REQUIRE_REVIEW"

class EgressPurpose(Enum):
    ALERTING = "ALERTING"
    REPORTING = "REPORTING"

class ToolEffect(Enum):
    READ = "READ"
    PROPOSE = "PROPOSE"
    WRITE = "WRITE"

class GuardrailStatus(Enum):
    ALLOWED = "ALLOWED"
    BLOCKED = "BLOCKED"
    APPROVAL_REQUIRED = "APPROVAL_REQUIRED"
    REPAIRABLE = "REPAIRABLE"
    PII_PATTERN_DETECTED = "PII_PATTERN_DETECTED"
    POLICY_CHECK_REQUIRED = "POLICY_CHECK_REQUIRED"
    NEED_MORE_EVIDENCE = "NEED_MORE_EVIDENCE"

## Step 2: Content Classification

In [ ]:
class ContentItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    item_id: str
    tenant_id: str
    source_type: SourceType
    trust: TrustLevel
    payload: str

class InjectionSignal(BaseModel):
    detected: bool
    markers: List[str]
    ambiguous: bool = False

class ContentDecision(BaseModel):
    disposition: ContentDisposition
    signal: InjectionSignal
    reason: str

def detect_injection_signals(item: ContentItem) -> InjectionSignal:
    known_markers = ["ignore previous instructions", "restart production", "export customer records"]
    ambiguous_markers = ["ignore policy"]
    
    found = []
    text = item.payload.lower()
    
    detected_definitive = False
    for marker in known_markers:
        if marker in text:
            found.append(marker)
            detected_definitive = True
            
    ambiguous = False
    for marker in ambiguous_markers:
        if marker in text:
            found.append(marker)
            if not detected_definitive:
                ambiguous = True
            
    return InjectionSignal(detected=len(found) > 0, markers=found, ambiguous=ambiguous)

def classify_content(item: ContentItem) -> ContentDecision:
    signal = detect_injection_signals(item)
    if signal.detected:
        if signal.ambiguous or item.trust == TrustLevel.TRUSTED:
            return ContentDecision(disposition=ContentDisposition.REQUIRE_REVIEW, signal=signal, reason="Ambiguous markers. Requires review.")
        else:
            return ContentDecision(disposition=ContentDisposition.QUARANTINE, signal=signal, reason="Injection detected in untrusted content.")
        
    if item.trust == TrustLevel.TRUSTED:
        return ContentDecision(disposition=ContentDisposition.ALLOW_AS_DATA, signal=signal, reason="Trusted content.")
        
    return ContentDecision(
        disposition=ContentDisposition.ALLOW_AS_DATA,
        signal=signal,
        reason="Eligible for delimited inclusion under downstream containment controls."
    )

### Why REQUIRE_REVIEW?
Detection signals can have false positives (e.g., "ignore policy" in an internal legitimate document). `REQUIRE_REVIEW` prevents heuristic detection from becoming an automatic authority decision, keeping humans in the loop for ambiguous cases.

## Step 3: Tool and Authority Validation

In [ ]:
class ExecutionContext(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tenant_id: str
    user_id: str
    environment: str
    approved_capabilities: List[str]
    allowed_destinations: List[str]
    request_id: str
    policy_version: str

class ValidatedApprovalContext(BaseModel):
    model_config = ConfigDict(extra="forbid")
    action: str
    tenant: str
    target_digest: str
    expiry: datetime
    policy_version: str

class ToolCall(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str
    arguments: dict
    requested_tenant_id: Optional[str] = None

class ToolDecision(BaseModel):
    status: GuardrailStatus
    reason: str

# Strict Schemas
class RestartServiceArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    cluster: Literal["us-east", "eu-west", "ap-south"]

class ExportCustomerRecordsArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    destination: str
    purpose: EgressPurpose
    sensitivity: Sensitivity

class ToolDefinition(BaseModel):
    name: str
    required_permissions: List[str]
    requires_approval: bool
    allowed_environments: List[str]
    input_model: Type[BaseModel]

TOOL_REGISTRY = {
    "restart_service": ToolDefinition(
        name="restart_service", required_permissions=["deployment:write"], requires_approval=True,
        allowed_environments=["production"], input_model=RestartServiceArgs
    ),
    "export_customer_records": ToolDefinition(
        name="export_customer_records", required_permissions=["customer_data:export"], requires_approval=True,
        allowed_environments=["production"], input_model=ExportCustomerRecordsArgs
    )
}

## Step 4: Core Gateway Logic

In [ ]:
def compute_digest(tool_name: str, tenant: str, arguments: dict) -> str:
    payload = {"tool_name": tool_name, "tenant": tenant, "arguments": arguments}
    canonical = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(canonical.encode()).hexdigest()

def validate_tool_call(call: ToolCall, context: ExecutionContext, validated_approval: Optional[ValidatedApprovalContext] = None) -> ToolDecision:
    if call.name not in TOOL_REGISTRY:
        return ToolDecision(status=GuardrailStatus.BLOCKED, reason=f"UNKNOWN_TOOL: {call.name}")
        
    definition = TOOL_REGISTRY[call.name]
    
    # Structural Validation
    try:
        validated_args = definition.input_model.model_validate(call.arguments)
    except ValidationError as e:
        repairs = [{"field": err.get("loc"), "error_code": err.get("type"), "repair_hint": err.get("msg")} for err in e.errors()]
        return ToolDecision(status=GuardrailStatus.REPAIRABLE, reason=json.dumps(repairs))
    
    # Authority Validation
    if call.requested_tenant_id and call.requested_tenant_id != context.tenant_id:
        return ToolDecision(status=GuardrailStatus.BLOCKED, reason="WRONG_TENANT")
        
    for perm in definition.required_permissions:
        if perm not in context.approved_capabilities:
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="UNAUTHORIZED_CAPABILITY")
            
    if context.environment not in definition.allowed_environments:
        return ToolDecision(status=GuardrailStatus.BLOCKED, reason="ENVIRONMENT_NOT_ALLOWED")
        
    # Approval Validation
    if definition.requires_approval:
        if not validated_approval:
            return ToolDecision(status=GuardrailStatus.APPROVAL_REQUIRED, reason="APPROVAL_REQUIRED")
        
        if validated_approval.tenant != context.tenant_id:
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="APPROVAL_MISMATCH: wrong tenant")
        if validated_approval.action != call.name:
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="APPROVAL_MISMATCH: wrong action")
        if validated_approval.policy_version != context.policy_version:
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="APPROVAL_MISMATCH: stale policy version")
        if validated_approval.expiry <= datetime.now(timezone.utc):
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="APPROVAL_MISMATCH: expired approval")
            
        expected_digest = compute_digest(call.name, context.tenant_id, validated_args.model_dump(mode="json"))
        if validated_approval.target_digest != expected_digest:
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="APPROVAL_MISMATCH: wrong target digest")

    # Integrated Egress Validation
    if call.name == "export_customer_records":
        if validated_args.destination not in context.allowed_destinations:
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="EGRESS_DENIED: Unapproved destination")
        if validated_args.sensitivity == Sensitivity.RESTRICTED and validated_args.destination != "secure-vault@northstar.internal":
            return ToolDecision(status=GuardrailStatus.BLOCKED, reason="EGRESS_DENIED: Restricted data destination mismatch")
        
    return ToolDecision(status=GuardrailStatus.ALLOWED, reason="ALLOW")

## Step 5: Execute Adversarial Tests

In [ ]:
# Setup
context = ExecutionContext(
    tenant_id="northstar", user_id="user_123", environment="production",
    approved_capabilities=["deployment:write", "customer_data:export"],
    allowed_destinations=["alerts@northstar.internal", "secure-vault@northstar.internal"],
    request_id="req_999", policy_version="1.0"
)

# Test 1: Bypassed injection caught by authorization
bad_call = ToolCall(name="restart_service", arguments={"cluster": "eu-west"})
print("Test 1 (Bypass Caught):", validate_tool_call(bad_call, context, validated_approval=None).status)

# Test 2: Valid approval succeeds
args = {"cluster": "eu-west"}
digest = compute_digest("restart_service", "northstar", args)
approval = ValidatedApprovalContext(
    action="restart_service", tenant="northstar", target_digest=digest,
    expiry=datetime.now(timezone.utc) + timedelta(minutes=10), policy_version="1.0"
)
good_call = ToolCall(name="restart_service", arguments=args)
print("Test 2 (Valid Approval):", validate_tool_call(good_call, context, validated_approval=approval).status)

# Test 3: Egress block (invalid purpose)
# Simulating an exfiltration attempt disguised as a report.
# The schema validates EgressPurpose which only allows ALERTING and REPORTING.
# If an attacker hallucinated "exfiltration_test", it yields REPAIRABLE (schema failure).
egress_call = ToolCall(name="export_customer_records", arguments={
    "destination": "alerts@northstar.internal", "purpose": "exfiltration_test", "sensitivity": "INTERNAL"
})
print("Test 3 (Egress Block):", validate_tool_call(egress_call, context, validated_approval=None).status)
